# Chapter 14 Lab — Question Answering and Semantic Search

A small open-domain QA pipeline: TF-IDF retrieval vs. dense (Sentence-BERT) retrieval on
paraphrased queries, then an extractive-QA reader over the top-retrieved passage. Corpus is a
few dozen short, self-written reference excerpts — not a copyrighted book corpus.

## 1. A small reference corpus

In [ ]:
passages = [
    "The Transformer architecture was introduced in the 2017 paper 'Attention Is All You Need'.",
    "Paris is the capital of France and sits on the Seine river.",
    "BERT is a masked-language-model encoder released by Google in 2018.",
    "GPT models are trained with a causal, next-token-prediction objective.",
    "The Eiffel Tower was completed in 1889 for the World's Fair in Paris.",
    "Retrieval-Augmented Generation combines a retriever with a generative model.",
]

## 2. Sparse (TF-IDF) retrieval

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vec = TfidfVectorizer().fit(passages)
p_vecs = vec.transform(passages)

def sparse_retrieve(query, k=2):
    q = vec.transform([query])
    sims = cosine_similarity(q, p_vecs)[0]
    top = sims.argsort()[::-1][:k]
    return [(passages[i], sims[i]) for i in top]

for p, s in sparse_retrieve("When was the Eiffel Tower built?"):
    print(round(s, 3), p)

## 3. Dense (semantic) retrieval — catches paraphrase sparse retrieval misses

In [ ]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-MiniLM-L6-v2")
p_embs = model.encode(passages, convert_to_tensor=True)

def dense_retrieve(query, k=2):
    q_emb = model.encode(query, convert_to_tensor=True)
    sims = util.cos_sim(q_emb, p_embs)[0]
    top = sims.argsort(descending=True)[:k]
    return [(passages[i], float(sims[i])) for i in top]

paraphrase_query = "What year did the monument in Paris finish construction?"
print("Sparse:", sparse_retrieve(paraphrase_query))
print("Dense :", dense_retrieve(paraphrase_query))

## 4. Extractive-QA reader over the top-retrieved passage

In [ ]:
from transformers import pipeline
qa = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")

question = "When was the Transformer introduced?"
top_passage, _ = dense_retrieve(question, k=1)[0]
print(qa(question=question, context=top_passage))

## Exercise

Add a query with an exact product code or ID-like token (e.g. "model XJ-42") to `passages` and
a matching query. Compare sparse vs. dense retrieval on it — which one finds it reliably, and
why does that match the hybrid-retrieval argument in the chapter text?